# Precio y Promoción
### Elasticidad, codificación y el impacto en pesos

`Fase 2 · Video 10` — serie **Forecasting de Ventas de la A a la Z**

Cierre de la **Fase 2** con la palanca más poderosa del retail: el precio. Medimos la **elasticidad**
(y por qué la estimación ingenua engaña), codificamos promociones y traducimos el error a **dinero**.

---
## 1. Librerías y carga de datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, PercentFormatter
import statsmodels.formula.api as smf
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
money_fmt = FuncFormatter(lambda x, _: f'${x:,.0f}')
print('✅ Librerías cargadas')

In [ ]:
def find_csv(filename, max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv('sales_history.csv')
df = pd.read_csv(csv_path, parse_dates=['date']).sort_values('date')
catalog = pd.read_csv(find_csv('sku_catalog.csv')).set_index('sku_id')

# SKU representativo: el de mayor revenue entre los de demanda regular
regulars = catalog.index[catalog['demand_profile'] == 'Regular']
SKU = df[df['sku_id'].isin(regulars)].groupby('sku_id')['revenue'].sum().idxmax()

print(f'✅ {df["sku_id"].nunique()} SKUs | días en promo: {df["on_promo"].mean():.1%}')
print(f'   Descuento medio en promo: {df.loc[df["on_promo"]==1, "discount"].mean():.0%}')
print(f'   SKU de referencia para el análisis: {SKU} ({catalog.loc[SKU, "category"]})')

---
## 2. Anatomía de una promoción

Una promoción es un **descuento temporal de precio**. Al graficar precio y demanda del mismo SKU, el
patrón salta: cuando el precio cae (barras naranjas = promo), la demanda se dispara. Ese acoplamiento
precio↔demanda es lo que vamos a cuantificar.

In [ ]:
sub = (df[df['sku_id'] == SKU]
       .groupby('date')
       .agg(units=('units_sold', 'sum'), price=('unit_price', 'mean'),
            on_promo=('on_promo', 'max'), discount=('discount', 'max'))
       .asfreq('D'))

w = sub.loc['2023']

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(w.index, w['units'], color=BLUE, linewidth=1.2, label='demanda (unidades)')
for d in w.index[w['on_promo'] == 1]:
    ax1.axvspan(d, d + pd.Timedelta(days=1), color=ORANGE, alpha=0.18)
ax1.set_ylabel('Unidades', color=BLUE)
ax1.set_title(f'{SKU} — demanda y precio en 2023 (naranja = promoción)', fontsize=13, fontweight='bold')

ax2 = ax1.twinx()
ax2.plot(w.index, w['price'], color=RED, linewidth=1.5, label='precio')
ax2.set_ylabel('Precio unitario', color=RED)
ax2.grid(False)
plt.tight_layout()
plt.show()

lift = sub.loc[sub['on_promo'] == 1, 'units'].mean() / sub.loc[sub['on_promo'] == 0, 'units'].mean() - 1
print(f'Demanda media en promo vs normal: {lift:+.0%}')

---
## 3. Elasticidad: la estimación ingenua engaña

La **elasticidad precio** es el % de cambio en la demanda ante un 1% de cambio en el precio. Con un modelo
de elasticidad constante, se estima como la pendiente de una regresión **log-log**:

$$\log Q = \alpha + \beta \, \log P + \varepsilon, \qquad \beta = \text{elasticidad}$$

El instinto es correr esa regresión directo sobre los datos. Veamos qué sale.

In [ ]:
d = df[(df['sku_id'] == SKU) & (df['units_sold'] > 0)].copy()
d['lq'] = np.log(d['units_sold'])
d['lp'] = np.log(d['unit_price'])

beta_naive = np.polyfit(d['lp'], d['lq'], 1)[0]

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(d['lp'], d['lq'], s=8, alpha=0.25, color=BLUE)
xs = np.linspace(d['lp'].min(), d['lp'].max(), 50)
ax.plot(xs, np.polyval(np.polyfit(d['lp'], d['lq'], 1), xs), color=RED, linewidth=2.5,
        label=f'pendiente (elasticidad) = {beta_naive:.2f}')
ax.set_xlabel('log(precio)'); ax.set_ylabel('log(demanda)')
ax.set_title(f'{SKU} — regresión log-log INGENUA (todos los canales)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Elasticidad ingenua estimada: {beta_naive:.2f}')
print(f'Elasticidad REAL (inyectada): {catalog.loc[SKU, "elasticity"]:.2f}')
print('→ ¡La ingenua está sesgada hacia 0! Muy lejos de la verdad. ¿Por qué?')

---
## 4. Controlar los confusores para recuperar la elasticidad real

La regresión ingenua mezcla dos fuentes de variación de precio con efectos opuestos sobre la demanda:

- **Canal**: los canales baratos (Mayorista) no venden proporcionalmente más unidades por su menor precio
  — su relación precio/volumen depende del canal, no de la elasticidad. Esto **sesga la pendiente hacia 0**.
- **Estacionalidad y eventos**: mueven la demanda por razones ajenas al precio, añadiendo ruido.

La solución es **controlar** esos factores: efectos fijos de canal y features de calendario. Así aislamos
la variación de precio *limpia* (sobre todo la de las promociones) y su efecto real.

In [ ]:
d['dow'] = d['date'].dt.dayofweek
d['month'] = d['date'].dt.month
true_e = catalog.loc[SKU, 'elasticity']

models = {
    'Ingenua (todo)':            'lq ~ lp',
    '+ efectos fijos de canal':  'lq ~ lp + C(channel)',
    '+ calendario (dow, mes)':   'lq ~ lp + C(channel) + C(dow) + C(month)',
}
est = {name: smf.ols(f, d).fit().params['lp'] for name, f in models.items()}

fig, ax = plt.subplots(figsize=(10, 5))
names = list(est.keys())
vals = [est[n] for n in names]
ax.barh(names, vals, color=[RED, ORANGE, GREEN])
ax.axvline(true_e, color='black', linestyle='--', linewidth=2, label=f'elasticidad real = {true_e:.2f}')
ax.set_xlabel('Elasticidad estimada')
ax.set_title(f'{SKU} — cada control acerca la estimación a la verdad', fontsize=13, fontweight='bold')
for i, v in enumerate(vals):
    ax.text(v, i, f' {v:.2f}', va='center', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

for n, v in est.items():
    print(f'  {n:<26}: {v:+.2f}')
print(f'  {"REAL (inyectada)":<26}: {true_e:+.2f}')
print('\n→ Al controlar canal + calendario, la estimación converge a la elasticidad real.')

---
## 5. Validación en todo el portafolio

¿Funciona el método más allá de un SKU? Estimamos la elasticidad **controlada** para cada SKU y la
comparamos con la verdad inyectada. Si el método es bueno, los puntos caen sobre la diagonal.

In [ ]:
def estimate_elasticity(sku):
    s = df[(df['sku_id'] == sku) & (df['units_sold'] > 0)].copy()
    if len(s) < 200 or s['unit_price'].nunique() < 10:
        return np.nan
    s['lq'] = np.log(s['units_sold']); s['lp'] = np.log(s['unit_price'])
    s['dow'] = s['date'].dt.dayofweek; s['month'] = s['date'].dt.month
    return smf.ols('lq ~ lp + C(channel) + C(dow) + C(month)', s).fit().params['lp']

res = catalog.copy()
res['elast_est'] = [estimate_elasticity(k) for k in res.index]
res = res.dropna(subset=['elast_est'])

fig, ax = plt.subplots(figsize=(8, 8))
cats = res['category'].unique()
palette = [BLUE, GREEN, ORANGE, PURPLE, RED]
for c, col in zip(sorted(cats), palette):
    m = res['category'] == c
    ax.scatter(res.loc[m, 'elasticity'], res.loc[m, 'elast_est'], s=55, alpha=0.8,
               color=col, edgecolor='white', label=c)
lims = [-3.0, -0.2]
ax.plot(lims, lims, color='black', linestyle='--', linewidth=1.5, label='estimada = real')
ax.set_xlabel('Elasticidad real (inyectada)'); ax.set_ylabel('Elasticidad estimada (controlada)')
ax.set_title('Validación de la estimación en el portafolio', fontsize=13, fontweight='bold')
ax.legend(title='Categoría', fontsize=9)
plt.tight_layout()
plt.show()

err = (res['elast_est'] - res['elasticity']).abs().mean()
corr = res['elasticity'].corr(res['elast_est'])
print(f'SKUs estimados: {len(res)}/{len(catalog)} (se excluyen intermitentes con pocos datos)')
print(f'Error absoluto medio: {err:.2f} | correlación estimada vs real: {corr:.2f}')
print('Las categorías caen donde deben: Alimentos (inelástica) arriba a la derecha,')
print('Electrónica/Deportes (elásticas) abajo a la izquierda.')

---
## 6. Codificación de promociones

Para que un modelo aproveche las promos hay que convertirlas en features. Tres codificaciones habituales:

- **One-hot** (`on_promo`): ¿hay promo hoy? (sí/no). Lo más simple.
- **Intensidad** (`discount`): *cuánto* es el descuento. Captura que −40% pega más que −10%.
- **Decaimiento** (*decay*): presión promocional reciente con memoria exponencial — útil cuando una promo
  deja efectos que se arrastran (o cae la demanda después, por *pull-forward*).

Medimos qué tanto se asocia cada codificación con la demanda destendenciada.

In [ ]:
s = sub.copy()
s['units'] = s['units'].fillna(0)
s['discount'] = s['discount'].fillna(0.0)
s['on_promo'] = s['on_promo'].fillna(0)
det = s['units'] / s['units'].rolling(31, center=True, min_periods=1).median()

# Decay: presión promocional con memoria exponencial (span 7 días)
s['promo_decay'] = s['discount'].ewm(span=7).mean()

enc = pd.DataFrame({
    'one_hot (on_promo)': s['on_promo'],
    'intensidad (discount)': s['discount'],
    'decay (EWM del descuento)': s['promo_decay'],
})
corrs = enc.corrwith(det)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(corrs.index, corrs.values, color=[BLUE, GREEN, ORANGE])
ax.set_xlabel('|correlación| con la demanda destendenciada')
ax.set_title('Qué codificación de promo captura más señal', fontsize=12, fontweight='bold')
for i, v in enumerate(corrs.values):
    ax.text(v, i, f' {v:.2f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

for n, v in corrs.items():
    print(f'  {n:<28}: {v:+.2f}')
print('\nLa intensidad supera al one-hot: distinguir −40% de −10% aporta. El decay capta menos')
print('aquí porque el dataset no modela pull-forward (canibalización post-promo); en datos')
print('reales el decay suele brillar justo por ese efecto de arrastre.')

---
## 7. El impacto en pesos de equivocarse

Aquí se cierra el círculo: un error de elasticidad no es un número académico, es **dinero**. Si planeas el
inventario de una promoción con la elasticidad **ingenua** (−0.99) en vez de la **real** (≈−1.94),
subestimas cuánto va a vender la promo → te quedas corto de stock → **ventas perdidas**.

In [ ]:
disc = 0.25                              # descuento típico
margin = 0.30                            # margen bruto supuesto
true_e = catalog.loc[SKU, 'elasticity']

base_units = sub.loc[sub['on_promo'] == 0, 'units'].median()      # demanda normal/día
promo_price = sub.loc[sub['on_promo'] == 1, 'price'].median()
promo_days_yr = (sub['on_promo'].sum() / 4)                       # días de promo al año

lift_true  = (1 - disc) ** true_e - 1
lift_naive = (1 - disc) ** beta_naive - 1
units_true  = base_units * (1 + lift_true)
units_naive = base_units * (1 + lift_naive)
faltante = units_true - units_naive                              # unidades no surtidas/día

costo_dia = faltante * promo_price * margin
costo_anual = costo_dia * promo_days_yr

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(['Plan con\nelasticidad ingenua', 'Demanda real\nde la promo'],
       [units_naive, units_true], color=[ORANGE, GREEN], edgecolor='white', width=0.55)
ax.set_ylabel('Unidades/día en promo')
ax.set_title(f'{SKU}: lo que planeas vs lo que ocurre (descuento {disc:.0%})',
             fontsize=12, fontweight='bold')
for i, v in enumerate([units_naive, units_true]):
    ax.text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontweight='bold')
ax.annotate('quiebre de stock', xy=(1, units_naive), xytext=(0.5, (units_true+units_naive)/2),
            color=RED, fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color=RED))
plt.tight_layout()
plt.show()

print(f'Elasticidad ingenua {beta_naive:.2f} → lift esperado {lift_naive:+.0%}')
print(f'Elasticidad real    {true_e:.2f} → lift real     {lift_true:+.0%}')
print(f'Faltante por día de promo : {faltante:,.0f} unidades')
print(f'Margen perdido por día     : ${costo_dia:,.0f}')
print(f'≈ {promo_days_yr:.0f} días de promo/año  →  COSTO ANUAL ≈ ${costo_anual:,.0f}  (¡en UN SOLO SKU!)')
print('Multiplicado por todo el portafolio, el error de elasticidad se vuelve material.')

---
## 8. Conclusiones y puente a la Fase 3

### Las reglas que te llevas

1. **El precio mueve la demanda.** La elasticidad es la palanca: modélala, no la ignores.
2. **La elasticidad ingenua engaña.** La regresión log-log directa está sesgada por el canal y la
   estacionalidad. **Controla los confusores** (efectos fijos + calendario) para recuperar la verdad.
3. **Codifica las promos con intensidad, no solo one-hot.** Distinguir −40% de −10% aporta señal; el decay
   ayuda cuando hay arrastre/canibalización.
4. **Traduce el error a pesos.** Un error de elasticidad se convierte en quiebres de stock o sobre-inventario.
   Es el lenguaje que entiende el negocio.

### Fin de la Fase 2

Ya tienes las tres familias de features: **calendario** (Video 8), **memoria** (lags/ventanas, Video 9) y
**precio/promoción** (este video). Con eso, el dataset está listo para modelar.

---

### Próximo video
**Video 12 — Baselines inteligentes: Naive, Seasonal Naive y Holt-Winters**
Arrancamos la Fase 3. Antes de cualquier modelo "fancy", establecemos el benchmark: por qué un
*seasonal naive* bien hecho ya le gana al 40% de los modelos que ves en LinkedIn.